# GPU Utilization Debug Notebook for GraphSAGE Training

This notebook helps identify and resolve GPU underutilization when training GraphSAGE models.

**Sections:**
1. Environment and Baseline Profiling
2. Data Loading Analysis
3. Batch Analysis
4. Model Forward Pass Profiling
5. Optimized Training Loop
6. Summary and Recommendations

## Section 1: Environment and Baseline Profiling

In [ ]:
# =============================================================================
# IMPORTS AND PATH SETUP
# =============================================================================

import sys
import time
import gc
from pathlib import Path
from datetime import datetime

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('.')

sys.path.insert(0, str(BASE_PATH))

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool

# Import from models.py
from models import (
    GraphSAGEModel,
    GraphSAGE,
    DatasetCache,
)

print(f"BASE_PATH: {BASE_PATH}")
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# =============================================================================
# GPU DIAGNOSTICS
# =============================================================================

def print_gpu_info():
    """Print comprehensive GPU information."""
    print("="*60)
    print("GPU DIAGNOSTICS")
    print("="*60)
    
    if not torch.cuda.is_available():
        print("CUDA is NOT available! Training will be on CPU.")
        return False
    
    print(f"\nCUDA Available: {torch.cuda.is_available()}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"PyTorch Version: {torch.__version__}")
    
    print(f"\n--- GPU Device Info ---")
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    
    # Get device properties
    props = torch.cuda.get_device_properties(0)
    print(f"\n--- GPU Specifications ---")
    print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
    print(f"Multi-Processor Count: {props.multi_processor_count}")
    print(f"Compute Capability: {props.major}.{props.minor}")
    
    # Memory info
    print(f"\n--- Current Memory Usage ---")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.3f} GB")
    print(f"Cached: {torch.cuda.memory_reserved(0) / 1024**3:.3f} GB")
    
    # Check if TF32 is enabled (Ampere+ GPUs)
    print(f"\n--- Performance Settings ---")
    print(f"TF32 for MatMul: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"TF32 for cuDNN: {torch.backends.cudnn.allow_tf32}")
    print(f"cuDNN Benchmark: {torch.backends.cudnn.benchmark}")
    
    return True

gpu_available = print_gpu_info()
device = torch.device("cuda" if gpu_available else "cpu")
print(f"\nUsing device: {device}")

In [ ]:
# =============================================================================
# ENABLE PERFORMANCE OPTIMIZATIONS
# =============================================================================

if torch.cuda.is_available():
    # Enable TF32 for faster computation on Ampere+ GPUs (RTX 30xx, A100, etc.)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    # Enable cuDNN autotuner - finds optimal algorithms for your specific GPU
    torch.backends.cudnn.benchmark = True
    
    print("Performance optimizations enabled:")
    print(f"  TF32 MatMul: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"  TF32 cuDNN: {torch.backends.cudnn.allow_tf32}")
    print(f"  cuDNN Benchmark: {torch.backends.cudnn.benchmark}")

In [ ]:
# =============================================================================
# GPU UTILIZATION MONITORING UTILITIES
# =============================================================================

class GPUMonitor:
    """Simple GPU utilization tracker."""
    
    def __init__(self):
        self.has_pynvml = False
        try:
            import pynvml
            pynvml.nvmlInit()
            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            self.has_pynvml = True
            print("pynvml initialized - GPU utilization monitoring available")
        except:
            print("pynvml not available - install with: pip install pynvml")
            print("GPU utilization will be estimated from memory usage only")
    
    def get_utilization(self):
        """Get GPU utilization percentage."""
        if self.has_pynvml:
            import pynvml
            util = pynvml.nvmlDeviceGetUtilizationRates(self.handle)
            return {'gpu': util.gpu, 'memory': util.memory}
        return None
    
    def get_memory_info(self):
        """Get GPU memory info."""
        if torch.cuda.is_available():
            return {
                'allocated_gb': torch.cuda.memory_allocated(0) / 1024**3,
                'cached_gb': torch.cuda.memory_reserved(0) / 1024**3,
                'max_allocated_gb': torch.cuda.max_memory_allocated(0) / 1024**3,
            }
        return None
    
    def reset_peak_memory(self):
        """Reset peak memory tracking."""
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(0)

gpu_monitor = GPUMonitor()

In [ ]:
# =============================================================================
# TIMING UTILITIES
# =============================================================================

class CUDATimer:
    """Accurate GPU timing using CUDA events."""
    
    def __init__(self):
        self.use_cuda = torch.cuda.is_available()
        if self.use_cuda:
            self.start_event = torch.cuda.Event(enable_timing=True)
            self.end_event = torch.cuda.Event(enable_timing=True)
        self.cpu_start = None
    
    def start(self):
        if self.use_cuda:
            self.start_event.record()
        self.cpu_start = time.perf_counter()
    
    def stop(self):
        cpu_time = time.perf_counter() - self.cpu_start
        if self.use_cuda:
            self.end_event.record()
            torch.cuda.synchronize()
            gpu_time = self.start_event.elapsed_time(self.end_event) / 1000.0  # ms to s
            return {'gpu_time': gpu_time, 'cpu_time': cpu_time}
        return {'cpu_time': cpu_time}

def benchmark_fn(fn, n_runs=10, warmup=3):
    """Benchmark a function with proper warmup."""
    timer = CUDATimer()
    
    # Warmup
    for _ in range(warmup):
        fn()
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    # Benchmark
    times = []
    for _ in range(n_runs):
        timer.start()
        fn()
        result = timer.stop()
        times.append(result.get('gpu_time', result['cpu_time']))
    
    return {
        'mean': np.mean(times),
        'std': np.std(times),
        'min': np.min(times),
        'max': np.max(times),
    }

print("Timing utilities ready.")

## Section 2: Data Loading Analysis

This section profiles:
- Dataset loading time from cache
- DataLoader iteration speed with different `num_workers`
- Impact of `persistent_workers` and `prefetch_factor`

In [ ]:
# =============================================================================
# LOAD DATASET FROM CACHE
# =============================================================================

# Choose which dataset to use for debugging (d3 is fastest, d7 is more representative)
DATASET_NAME = "d5_baseline"  # Options: d3_baseline, d5_baseline, d7_baseline, d9_baseline

print(f"Loading dataset: {DATASET_NAME}")
print("="*60)

# Time the dataset loading
load_start = time.perf_counter()

cache = DatasetCache(base_path=BASE_PATH, device=device)
cache.load(DATASET_NAME)
graphs = cache.get_graphs()

load_time = time.perf_counter() - load_start

print(f"\nDataset loaded in {load_time:.2f} seconds")
print(f"Total graphs: {len(graphs):,}")
print(f"Loading speed: {len(graphs) / load_time:,.0f} graphs/second")

In [ ]:
# =============================================================================
# BENCHMARK DATALOADER WITH DIFFERENT num_workers
# =============================================================================

def benchmark_dataloader(graphs, batch_size, num_workers, n_batches=100, 
                         persistent_workers=False, prefetch_factor=2):
    """
    Benchmark DataLoader iteration speed.
    
    Returns time to iterate through n_batches.
    """
    loader_kwargs = {
        'batch_size': batch_size,
        'shuffle': True,
        'num_workers': num_workers,
        'pin_memory': True if torch.cuda.is_available() else False,
    }
    
    # persistent_workers only valid when num_workers > 0
    if num_workers > 0:
        loader_kwargs['persistent_workers'] = persistent_workers
        loader_kwargs['prefetch_factor'] = prefetch_factor
    
    loader = DataLoader(graphs, **loader_kwargs)
    
    # Warmup: iterate through a few batches
    loader_iter = iter(loader)
    for _ in range(min(5, n_batches)):
        try:
            batch = next(loader_iter)
            if torch.cuda.is_available():
                batch = batch.to(device)
        except StopIteration:
            loader_iter = iter(loader)
    
    # Benchmark
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start = time.perf_counter()
    batch_count = 0
    total_graphs = 0
    
    loader_iter = iter(loader)
    while batch_count < n_batches:
        try:
            batch = next(loader_iter)
            if torch.cuda.is_available():
                batch = batch.to(device, non_blocking=True)
            batch_count += 1
            total_graphs += batch.num_graphs
        except StopIteration:
            loader_iter = iter(loader)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    elapsed = time.perf_counter() - start
    
    return {
        'time': elapsed,
        'batches_per_sec': batch_count / elapsed,
        'graphs_per_sec': total_graphs / elapsed,
    }

# Test different num_workers configurations
print("="*60)
print("DATALOADER BENCHMARK")
print("="*60)
print(f"\nBatch size: 256, Testing {100} batches each\n")

# Use a subset for faster benchmarking
subset_size = min(100000, len(graphs))
benchmark_graphs = graphs[:subset_size]
print(f"Using {subset_size:,} graphs for benchmarking\n")

worker_configs = [0, 2, 4, 8]
results = []

for num_workers in worker_configs:
    print(f"Testing num_workers={num_workers}...", end=" ", flush=True)
    try:
        result = benchmark_dataloader(
            benchmark_graphs, 
            batch_size=256, 
            num_workers=num_workers,
            n_batches=100,
            persistent_workers=(num_workers > 0),
            prefetch_factor=2
        )
        result['num_workers'] = num_workers
        results.append(result)
        print(f"{result['batches_per_sec']:.1f} batches/sec, {result['graphs_per_sec']:,.0f} graphs/sec")
    except Exception as e:
        print(f"Failed: {e}")

# Find best configuration
if results:
    best = max(results, key=lambda x: x['graphs_per_sec'])
    baseline = results[0]  # num_workers=0
    print(f"\n{'='*60}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"Best: num_workers={best['num_workers']} -> {best['graphs_per_sec']:,.0f} graphs/sec")
    print(f"Speedup vs num_workers=0: {best['graphs_per_sec'] / baseline['graphs_per_sec']:.2f}x")

In [ ]:
# =============================================================================
# VISUALIZE DATALOADER RESULTS
# =============================================================================

if results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    workers = [r['num_workers'] for r in results]
    graphs_per_sec = [r['graphs_per_sec'] for r in results]
    batches_per_sec = [r['batches_per_sec'] for r in results]
    
    # Bar chart for graphs/sec
    bars = ax1.bar(range(len(workers)), graphs_per_sec, color='steelblue')
    ax1.set_xticks(range(len(workers)))
    ax1.set_xticklabels([str(w) for w in workers])
    ax1.set_xlabel('num_workers')
    ax1.set_ylabel('Graphs per second')
    ax1.set_title('DataLoader Throughput vs num_workers')
    
    # Add value labels
    for bar, val in zip(bars, graphs_per_sec):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
                 f'{val:,.0f}', ha='center', va='bottom', fontsize=9)
    
    # Speedup chart
    baseline = graphs_per_sec[0]
    speedups = [g / baseline for g in graphs_per_sec]
    bars2 = ax2.bar(range(len(workers)), speedups, color='seagreen')
    ax2.set_xticks(range(len(workers)))
    ax2.set_xticklabels([str(w) for w in workers])
    ax2.set_xlabel('num_workers')
    ax2.set_ylabel('Speedup vs num_workers=0')
    ax2.set_title('DataLoader Speedup')
    ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
    
    for bar, val in zip(bars2, speedups):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
                 f'{val:.2f}x', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("\n⚠️  FINDING: If num_workers=0 is nearly as fast as num_workers>0,")
    print("   the bottleneck is NOT data loading but likely the model forward pass.")

## Section 3: Batch Analysis

This section analyzes:
- Graph sizes (nodes and edges per graph)
- Impact of batch size on GPU utilization
- Memory usage vs batch size tradeoffs